# 04 - Lean Model-Readiness EDA

This notebook is intentionally compact. It checks only what is needed before building the final model-ready datasets:

1. Is the processed table clean and split correctly?
2. Which columns are targets, predictors, or quality metadata?
3. Are target candidates and source uncertainty still important?
4. What should the next notebook export?

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    display
except NameError:
    def display(obj):
        if hasattr(obj, 'to_string'):
            print(obj.to_string())
        else:
            print(obj)

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sea_rainfall_daily_2020_2025_processed_features.csv'
REPORT_DIR = PROJECT_ROOT / 'reports' / '04_processed_model_readiness_eda'
FIGURE_DIR = REPORT_DIR / 'figures'
TABLE_DIR = REPORT_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLS = [
    'target_nasa_power_precipitation_mm',
    'target_open_meteo_precipitation_mm',
    'target_baseline_two_source_mean_mm',
    'target_nasa_reference_consensus_mm',
    'target_open_meteo_reference_consensus_mm',
]

QUALITY_METADATA_COLS = [
    'target_reference_consensus_band_mm',
    'source_bias_nasa_minus_open_meteo_mm',
    'source_abs_diff_precipitation_mm',
    'source_relative_abs_diff_precipitation',
    'source_bias_train_city_month_mm',
    'wet_day_disagreement',
    'high_source_gap_10mm',
    'high_source_gap_20mm',
]

STRICT_FEATURES = [
    'canonical_latitude', 'canonical_longitude',
    'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos',
    'rainfall_lag_1d_mm', 'rainfall_lag_7d_mm', 'rainfall_lag_30d_mm',
    'rainfall_rolling_7d_mean_prev_mm', 'rainfall_rolling_30d_sum_prev_mm', 'rainfall_rolling_90d_mean_prev_mm',
    'wet_spell_days_prev', 'dry_spell_days_prev', 'was_wet_previous_day',
]

WEATHER_FEATURES = [
    'temp_mean_c_mean_two_sources',
    'temp_max_c_mean_two_sources',
    'temp_min_c_mean_two_sources',
    'relative_humidity_pct_mean_two_sources',
    'wind_speed_ms_mean_two_sources',
    'surface_pressure_kpa_mean_two_sources',
]

IMPUTATION_FLAGS = [column for column in [
    'rainfall_lag_1d_mm_was_imputed',
    'rainfall_lag_7d_mm_was_imputed',
    'rainfall_lag_30d_mm_was_imputed',
    'rainfall_rolling_7d_mean_prev_mm_was_imputed',
    'rainfall_rolling_30d_sum_prev_mm_was_imputed',
    'rainfall_rolling_90d_mean_prev_mm_was_imputed',
    'wet_spell_days_prev_was_imputed',
    'dry_spell_days_prev_was_imputed',
    'was_wet_previous_day_was_imputed',
] if column in pd.read_csv(PROCESSED_PATH, nrows=0).columns]

plt.rcParams.update({
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.22,
    'font.family': 'DejaVu Sans',
})


def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved table: {path.relative_to(PROJECT_ROOT)}')


def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved figure: {path.relative_to(PROJECT_ROOT)}')

## 1. Load and Basic Quality Check

This is only a gate check: row count, column count, split count, missing values, and negative targets.

In [ ]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=['date']).sort_values(['entity_id', 'date']).reset_index(drop=True)

quality_summary = pd.DataFrame([
    {'check': 'rows', 'value': len(df), 'expected': 26304, 'passed': len(df) == 26304},
    {'check': 'columns', 'value': df.shape[1], 'expected': 'compact schema', 'passed': df.shape[1] <= 70},
    {'check': 'entities', 'value': df['entity_id'].nunique(), 'expected': 12, 'passed': df['entity_id'].nunique() == 12},
    {'check': 'missing_cells', 'value': int(df.isna().sum().sum()), 'expected': 0, 'passed': int(df.isna().sum().sum()) == 0},
    {'check': 'negative_target_rows', 'value': int((df[TARGET_COLS] < 0).any(axis=1).sum()), 'expected': 0, 'passed': int((df[TARGET_COLS] < 0).any(axis=1).sum()) == 0},
])

split_summary = (
    df.groupby('split', as_index=False)
    .agg(rows=('entity_id', 'size'), start_date=('date', 'min'), end_date=('date', 'max'), entities=('entity_id', 'nunique'))
)

save_table(quality_summary, '01_quality_summary.csv')
save_table(split_summary, '02_split_summary.csv')
display(quality_summary)
display(split_summary)

## 2. Minimal Column Role Table

The next notebook needs to know which columns are targets, usable predictors, and metadata. This table is intentionally simple.

In [ ]:
def role(column):
    if column in TARGET_COLS:
        return 'target_candidate'
    if column in QUALITY_METADATA_COLS:
        return 'quality_metadata_not_predictor'
    if column in STRICT_FEATURES:
        return 'strict_forecast_predictor'
    if column in WEATHER_FEATURES:
        return 'weather_assisted_predictor'
    if column in IMPUTATION_FLAGS:
        return 'imputation_flag'
    if column in ['entity_id', 'date', 'split', 'country', 'location_name']:
        return 'identifier'
    return 'metadata_or_review'


column_roles = pd.DataFrame({
    'column_name': df.columns,
    'role': [role(column) for column in df.columns],
    'dtype': [str(df[column].dtype) for column in df.columns],
})
role_summary = column_roles['role'].value_counts().rename_axis('role').reset_index(name='column_count')

save_table(column_roles, '03_column_roles.csv')
save_table(role_summary, '04_role_summary.csv')
display(role_summary)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
plot_data = role_summary.sort_values('column_count')
ax.barh(plot_data['role'], plot_data['column_count'], color='#4C78A8')
ax.set_xlabel('Column count')
ax.set_ylabel('Role')
ax.set_title('Processed Column Roles')
save_figure(fig, '01_column_roles.png')
plt.show()

## 3. Targets and Uncertainty: Enough to Decide the Next Step

Only two checks are needed here:

- target candidates are not identical, so keep them for sensitivity analysis;
- source uncertainty remains structured, so do not use uncertainty columns as ordinary predictors.

In [ ]:
target_summary = (
    df[TARGET_COLS]
    .agg(['mean', 'median', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'target_candidate'})
)

uncertainty_by_city = (
    df.groupby('location_name', as_index=False)
    .agg(
        mean_source_gap_mm=('source_abs_diff_precipitation_mm', 'mean'),
        high_gap_10mm_ratio=('high_source_gap_10mm', 'mean'),
        wet_day_disagreement_ratio=('wet_day_disagreement', 'mean'),
    )
    .sort_values('mean_source_gap_mm', ascending=False)
)

save_table(target_summary, '05_target_summary.csv')
save_table(uncertainty_by_city, '06_uncertainty_by_city.csv')
display(target_summary)
display(uncertainty_by_city)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
axes[0].barh(target_summary['target_candidate'], target_summary['mean'], color='#2A9D8F')
axes[0].set_xlabel('Mean rainfall (mm/day)')
axes[0].set_ylabel('Target candidate')
axes[0].set_title('Target candidates')

plot_unc = uncertainty_by_city.sort_values('mean_source_gap_mm')
axes[1].barh(plot_unc['location_name'], plot_unc['mean_source_gap_mm'], color='#E69F00')
axes[1].set_xlabel('Mean source gap (mm/day)')
axes[1].set_ylabel('City')
axes[1].set_title('Source uncertainty by city')

fig.suptitle('Target and Uncertainty Checks', fontsize=14, fontweight='bold', x=0.01, ha='left')
save_figure(fig, '02_target_and_uncertainty_checks.png')
plt.show()

## 4. Necessary Next Step

The next notebook should build final model-ready datasets, not another broad EDA.

In [ ]:
next_step_plan = pd.DataFrame([
    {
        'output_dataset': 'strict_forecast',
        'predictors': 'location + calendar + lag/rolling/spell features + imputation flags',
        'targets': 'all target candidates',
        'exclude': 'same-day weather and source-uncertainty fields',
        'why': 'lowest leakage risk',
    },
    {
        'output_dataset': 'weather_assisted_forecast',
        'predictors': 'strict_forecast predictors + weather summary features',
        'targets': 'all target candidates',
        'exclude': 'source-uncertainty fields as predictors',
        'why': 'use only if weather variables are available at prediction time',
    },
    {
        'output_dataset': 'uncertainty_metadata',
        'predictors': 'none',
        'targets': 'none',
        'exclude': 'not applicable',
        'why': 'use source uncertainty as weights, flags, or evaluation slices',
    },
])

save_table(next_step_plan, '07_next_step_plan.csv')
display(next_step_plan)

summary_lines = [
    '# Lean Model-Readiness Conclusions',
    '',
    f"- Processed table is clean: {len(df):,} rows, {df.shape[1]} columns, {int(df.isna().sum().sum())} missing cells.",
    '- Keep multiple target candidates for sensitivity analysis.',
    '- Do not use source uncertainty columns as ordinary predictors; keep them as metadata/weights/evaluation slices.',
    '- Next notebook should export only final model-ready views: `strict_forecast`, `weather_assisted_forecast`, and `uncertainty_metadata`.',
]
summary_path = REPORT_DIR / 'MODEL_READINESS_CONCLUSIONS_AND_NEXT_STEPS.md'
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')
print(f'Saved summary: {summary_path.relative_to(PROJECT_ROOT)}')